# STEP Files

Clear the OCP CAD Viewer first, then refresh and open the canonical `type2_scene.step` artifact in VS Code. Active plate-stack scenes include stub overhang bodies; metadata-only port sheets are still reconstructed on the AEDT side, not exported into STEP.


In [1]:
from __future__ import annotations



import gc
import sys

from IPython import get_ipython
from ocp_vscode import show_clear


def _clear_notebook_state() -> int:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("This cell must run inside an IPython kernel")

    namespace = ip.user_ns
    keep_names = {
        "__name__",
        "__doc__",
        "__package__",
        "__loader__",
        "__spec__",
        "__builtins__",
        "__builtin__",
        "get_ipython",
        "In",
        "Out",
        "_ih",
        "_oh",
        "_dh",
        "exit",
        "quit",
    }
    for name in tuple(namespace):
        if name in keep_names:
            continue
        del namespace[name]
        

    if "Out" in namespace:
        out_cache = namespace["Out"]
        if isinstance(out_cache, dict):
            out_cache.clear()
        

    for last_name in ("_", "__", "___"):
        if last_name in namespace:
            namespace[last_name] = None
        
    import sys
    for module_name in tuple(sys.modules):
        if module_name == "peetsfea" or module_name.startswith("peetsfea."):
            del sys.modules[module_name]
        import sys

    if "matplotlib.pyplot" in sys.modules:
        import matplotlib.pyplot as plt
        plt.close("all")
        
    from ocp_vscode import show_clear
    show_clear()
    import gc
    return gc.collect()


_collected = _clear_notebook_state()

Using port 3939


In [2]:
from pathlib import Path
import json
import subprocess
import sys
import tomllib


def _notebook_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    return Path(root_text).resolve()


def _object_table_by_id(tables: object, object_id: str, id_key: str) -> dict[str, object]:
    if not isinstance(tables, list):
        raise TypeError(f"{id_key} owner table must be a list")
    for table in tables:
        if not isinstance(table, dict):
            raise TypeError(f"{id_key} owner entries must be tables")
        if table[id_key] == object_id:
            return table
    raise KeyError(f"owner id not found: {object_id}")
def _selected_owner_table(payload: dict[str, object], owner_path: str) -> dict[str, object]:
    parts = owner_path.split(".")
    if len(parts) < 3:
        raise ValueError(f"owner path must have at least 3 parts: {owner_path}")
    root_name, object_id = parts[0], parts[1]
    if root_name == "modeled_objects":
        return _object_table_by_id(payload[root_name], object_id, "object_id")
    if root_name == "non_model_objects":
        return _object_table_by_id(payload[root_name], object_id, "id")
    raise ValueError(f"unsupported owner root: {root_name}")
def _selected_owner_value(payload: dict[str, object], owner_path: str) -> object:
    source_path = owner_path
    parts = source_path.split(".")
    table: object = _selected_owner_table(payload, source_path)
    for key in parts[2:]:
        if not isinstance(table, dict):
            raise TypeError(
                "owner source path segment is not a table: "
                f"owner_path={owner_path}, source_path={source_path}"
            )
        if key not in table:
            raise KeyError(
                "owner source path segment missing: "
                f"owner_path={owner_path}, source_path={source_path}, missing={key}"
            )
        table = table[key]
    if isinstance(table, dict) and "range" in table:
        raw_range = table["range"]
        if not isinstance(raw_range, list) or len(raw_range) != 4:
            raise ValueError(
                "range owner must be [is_integer, start, end, count]: "
                f"owner_path={owner_path}, source_path={source_path}"
            )
        if raw_range[1] != raw_range[2] or raw_range[3] != 1:
            raise ValueError(
                "selected sampled TOML owner is not frozen: "
                f"owner_path={owner_path}, source_path={source_path}"
            )
        return raw_range[1]
    return table


def show_selected_toml_parameters(sample_index: int) -> None:
    if not isinstance(sample_index, int):
        raise TypeError(f"sample_index must be int (actual={type(sample_index).__name__})")
    repo_root = _notebook_repo_root()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    from peetsfea.type2_spec_tools import type2_range_owner_descriptions

    if sample_index == -1:
        print("selected TOML mode: fixed example")
        print(f"TOML: {repo_root / 'examples' / 'type2_fixed.toml'}")
        return
    manifest_path = repo_root / "run" / "sampled" / "type2" / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    entries = manifest["entries"]
    if not isinstance(entries, list):
        raise TypeError("manifest.entries must be a list")
    selected_entries = [entry for entry in entries if entry["sample_index"] == sample_index]
    if len(selected_entries) != 1:
        raise ValueError(f"manifest must contain exactly one entry for sample_index={sample_index}")
    selected_entry = selected_entries[0]
    sampled_toml_path = Path(selected_entry["sampled_toml_path"])
    payload = tomllib.loads(sampled_toml_path.read_text(encoding="utf-8"))
    sampled = payload["sampled"]
    owner_paths = sampled["sampled_owner_paths"]
    owner_descriptions = type2_range_owner_descriptions(sampled_toml_path)
    print("selected TOML mode: sampled manifest")
    print(f"sample_index: {selected_entry['sample_index']}")
    print(f"design_id: {selected_entry['design_id']}")
    print(f"seed: {selected_entry['seed']}")
    print(f"retry: {selected_entry['retry_number']}")
    print(f"TOML: {sampled_toml_path}")
    print("sampled parameters:")
    for owner_path in owner_paths:
        description = owner_descriptions[owner_path]
        print(f"  {owner_path} = {_selected_owner_value(payload, owner_path)}  # {description}")


In [3]:

VIEW_INDEX = -1
BUILD_W_GUI = True
print(f"notebook state cleared; gc collected: {_collected}")
print(f"view index: {VIEW_INDEX}")
show_selected_toml_parameters(VIEW_INDEX)
del _collected


notebook state cleared; gc collected: 102
view index: -1
selected TOML mode: fixed example
TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml


In [4]:
from __future__ import annotations

from pathlib import Path
import json
import subprocess
import sys

import build123d as bd
from ocp_vscode import Camera, show, show_clear


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    repo_root = Path(root_text).resolve()
    pyproject_path = repo_root / "pyproject.toml"
    if not pyproject_path.is_file():
        raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
    return repo_root


REPO_ROOT = require_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from peetsfea.backend.pyaedt import type2_step_import_style as aedt_style
from peetsfea.type2_step_export import export_type2_step_artifacts


TYPE2_FIXED_SOURCE_TOML_PATH = REPO_ROOT / "examples" / "type2_fixed.toml"
TYPE2_FIXED_OUTPUT_DIR = REPO_ROOT / "run" / "step" / "type2"
TYPE2_FIXED_LEDGER_PATH = TYPE2_FIXED_OUTPUT_DIR / "type2_step_ledger.json"
TYPE2_SAMPLED_MANIFEST_PATH = REPO_ROOT / "run" / "sampled" / "type2" / "manifest.json"


def selected_manifest_entry_for_sample_index(sample_index: int) -> dict[str, object]:
    if not isinstance(sample_index, int):
        raise TypeError(f"sample_index must be int (actual={type(sample_index).__name__})")
    if sample_index < 0:
        raise ValueError(f"sampled manifest selection requires sample_index >= 0 (actual={sample_index})")
    manifest = json.loads(TYPE2_SAMPLED_MANIFEST_PATH.read_text(encoding="utf-8"))
    entries = manifest["entries"]
    if not isinstance(entries, list):
        raise TypeError("manifest.entries must be a list")
    selected_entries = [entry for entry in entries if entry["sample_index"] == sample_index]
    if len(selected_entries) != 1:
        raise ValueError(f"manifest must contain exactly one entry for sample_index={sample_index}")
    selected_entry = selected_entries[0]
    if not isinstance(selected_entry, dict):
        raise TypeError("selected manifest entry must be a table")
    return selected_entry


def alpha_from_transparency(transparency: float) -> float:
    if transparency < 0.0 or transparency > 1.0:
        raise ValueError(f"viewer transparency must be in [0, 1] (actual={transparency})")
    return 1.0 - transparency


_NON_MODEL_STYLE = (
    aedt_style._NON_MODEL_COLOR,
    alpha_from_transparency(aedt_style._NON_MODEL_TRANSPARENCY),
)
_PCB_STYLE = (
    aedt_style._TX_PCB_COLOR,
    alpha_from_transparency(aedt_style._TX_PCB_TRANSPARENCY),
)
_COPPER_STYLE = (
    aedt_style._TX_COPPER_COLOR,
    alpha_from_transparency(aedt_style._TX_COPPER_TRANSPARENCY),
)
_FERRITE_STYLE = (
    aedt_style._TX_UNDERLAY_FERRITE_COLOR,
    alpha_from_transparency(aedt_style._TX_UNDERLAY_FERRITE_TRANSPARENCY),
)
_PET_PSA_STYLE = (
    aedt_style._TX_UNDERLAY_PET_PSA_COLOR,
    alpha_from_transparency(aedt_style._TX_UNDERLAY_PET_PSA_TRANSPARENCY),
)
_AIR_STYLE = (
    aedt_style._TX_UNDERLAY_AIR_COLOR,
    alpha_from_transparency(aedt_style._TX_UNDERLAY_AIR_TRANSPARENCY),
)


def viewer_style_from_label(label: str) -> tuple[tuple[int, int, int], float]:
    if label.startswith(("tx_copper", "tx_inner_copper", "tx_outer_copper", "rx_copper", "tx_bridge", "rx_bridge", "tx_stub", "rx_stub")):
        return _COPPER_STYLE
    if label.startswith(("tx_pcb", "tx_inner_pcb", "tx_outer_pcb", "rx_pcb")):
        return _PCB_STYLE
    if label.startswith(aedt_style._UNDERLAY_FERRITE_NAME_PREFIXES):
        return _FERRITE_STYLE
    if label.startswith(aedt_style._UNDERLAY_PET_PSA_NAME_PREFIXES):
        return _PET_PSA_STYLE
    if label.startswith(aedt_style._UNDERLAY_AIR_NAME_PREFIXES):
        return _AIR_STYLE
    return _NON_MODEL_STYLE


def child_shapes(shape: bd.Shape) -> list[bd.Shape]:
    children = tuple(shape.children)
    if children:
        return list(children)
    return [shape]


def viewer_payload_for_shape(shape: bd.Shape, *, fallback_name: str) -> tuple[list[bd.Shape], list[str], list[tuple[int, int, int]], list[float]]:
    entries = child_shapes(shape)
    cad_objs: list[bd.Shape] = []
    names: list[str] = []
    colors: list[tuple[int, int, int]] = []
    alphas: list[float] = []
    for index, entry in enumerate(entries):
        entry_label = entry.label if isinstance(entry.label, str) and entry.label != "" else f"{fallback_name}_{index}"
        color, alpha = viewer_style_from_label(entry_label)
        cad_objs.append(entry)
        names.append(entry_label)
        colors.append(color)
        alphas.append(alpha)
    return (cad_objs, names, colors, alphas)


print(f"repo root: {REPO_ROOT}")
print(f"view index: {VIEW_INDEX}")
print(f"fixed type2 TOML: {TYPE2_FIXED_SOURCE_TOML_PATH}")
print(f"sample manifest: {TYPE2_SAMPLED_MANIFEST_PATH}")
show_clear()
print("viewer cleared")



repo root: /home/harry/Projects/PythonProjects/peetsfea-main
view index: -1
fixed type2 TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml
sample manifest: /home/harry/Projects/PythonProjects/peetsfea-main/run/sampled/type2/manifest.json
viewer cleared


## Refresh and show scene STEP

This notebook clears the viewer first, refreshes the fixed `type2_scene.step` artifact, and opens that single scene file with a reset camera. Plate-stack STEP scenes include shoe-fill and stub bodies; reconstructed port sheets remain metadata-only.


In [5]:
if VIEW_INDEX == -1:
    fixed_ledger = export_type2_step_artifacts(
        toml_path=TYPE2_FIXED_SOURCE_TOML_PATH,
        output_dir=TYPE2_FIXED_OUTPUT_DIR,
        ledger_path=TYPE2_FIXED_LEDGER_PATH,
        seed=0,
    )
    scene_step_path = Path(fixed_ledger["scene_step_path"])
    print("mode: fixed example")
    print(f"source TOML: {TYPE2_FIXED_SOURCE_TOML_PATH}")
    print(f"scene STEP: {scene_step_path}")
    print(f"ledger JSON: {TYPE2_FIXED_LEDGER_PATH}")
else:
    selected_entry = selected_manifest_entry_for_sample_index(sample_index=VIEW_INDEX)
    sampled_toml_path = Path(selected_entry["sampled_toml_path"])
    design_dir = Path(selected_entry["design_dir"])
    scene_step_path = Path(selected_entry["scene_step_path"])
    step_ledger_path = Path(selected_entry["step_ledger_path"])
    if not scene_step_path.is_file() or not step_ledger_path.is_file():
        generated_ledger = export_type2_step_artifacts(
            toml_path=sampled_toml_path,
            output_dir=design_dir,
            ledger_path=step_ledger_path,
            seed=selected_entry["seed"],
        )
        scene_step_path = Path(generated_ledger["scene_step_path"])
        print("generated sampled STEP for notebook view")
    print("mode: sampled manifest")
    print(f"manifest: {TYPE2_SAMPLED_MANIFEST_PATH}")
    print(f"sample index: {selected_entry['sample_index']}")
    print(f"design_id: {selected_entry['design_id']}")
    print(f"seed: {selected_entry['seed']}")
    print(f"retry: {selected_entry['retry_number']}")
    print(f"scene STEP: {scene_step_path}")
    print(f"ledger JSON: {step_ledger_path}")

shown_step = bd.import_step(scene_step_path)
cad_objs, names, colors, alphas = viewer_payload_for_shape(shown_step, fallback_name="type2_scene")
show(
    *cad_objs,
    names=names,
    colors=colors,
    alphas=alphas,
    transparent=True,
    reset_camera=Camera.RESET,
)



mode: fixed example
source TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml
scene STEP: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_scene.step
ledger JSON: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
+++++++++++++++++++++++


In [6]:
import subprocess
from typing import cast

from peetsfea.aedt import Hfss
from peetsfea.aedt.protocols import HfssSession
from peetsfea.backend.pyaedt.type2_step_setup_ready import setup_type2_step_ledger_into_hfss


def create_gui_hfss(design_name: str) -> HfssSession:
    return cast(HfssSession, Hfss(design=design_name, non_graphical=False, new_desktop=True, close_on_exit=False))


if "BUILD_W_GUI" not in globals():
    raise RuntimeError("Run the notebook state cell first so BUILD_W_GUI is defined")
if not isinstance(BUILD_W_GUI, bool):
    raise TypeError(f"BUILD_W_GUI must be bool (actual={type(BUILD_W_GUI).__name__})")
print(f"BUILD_W_GUI: {BUILD_W_GUI}")

if BUILD_W_GUI:
    if VIEW_INDEX == -1:
        fixed_ledger = export_type2_step_artifacts(
            toml_path=TYPE2_FIXED_SOURCE_TOML_PATH,
            output_dir=TYPE2_FIXED_OUTPUT_DIR,
            ledger_path=TYPE2_FIXED_LEDGER_PATH,
            seed=0,
        )
        output_aedt_path = TYPE2_FIXED_OUTPUT_DIR / "type2_fixed.aedt"
        imported_ledger_path = TYPE2_FIXED_OUTPUT_DIR / "type2_imported_ledger.json"
        print(f"fixed source TOML: {TYPE2_FIXED_SOURCE_TOML_PATH}")
        print(f"fixed STEP ledger: {TYPE2_FIXED_LEDGER_PATH}")
        print(f"fixed scene STEP: {fixed_ledger['scene_step_path']}")
        print("GUI fixed build delegated to setup_type2_step_ledger_into_hfss")
        hfss = create_gui_hfss("type2_fixed")
        result = setup_type2_step_ledger_into_hfss(
            hfss=hfss,
            step_ledger_path=TYPE2_FIXED_LEDGER_PATH,
            output_aedt_path=output_aedt_path,
            imported_ledger_path=imported_ledger_path,
            design_variables=(),
            run_aedt_design_validation=False,
        )
        print(f"fixed AEDT: {result['aedt_path']}")
    else:
        selected_entry = selected_manifest_entry_for_sample_index(sample_index=VIEW_INDEX)
        design_id = selected_entry["design_id"]
        command = [
            str(REPO_ROOT / ".venv" / "bin" / "python"),
            str(REPO_ROOT / "entry" / "build.py"),
            "--debug",
            "--manifest",
            str(TYPE2_SAMPLED_MANIFEST_PATH),
            "--design-id",
            design_id,
        ]
        print(f"manifest: {TYPE2_SAMPLED_MANIFEST_PATH}")
        print(f"debug design_id: {design_id}")
        print("GUI debug build delegated to entry/build.py")
        subprocess.run(command, cwd=REPO_ROOT / "run", check=True)
else:
    print("BUILD_W_GUI is False; GUI build skipped")


BUILD_W_GUI: True
fixed source TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml
fixed STEP ledger: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
fixed scene STEP: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_scene.step
GUI fixed build delegated to setup_type2_step_ledger_into_hfss
PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO: PyAEDT version 0.25.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: AEDT version 2025.2.
PyAEDT INFO: New AEDT session is starting on gRPC port 42571.
PyAEDT INFO: Starting new AEDT gRPC session on port 42571.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: TransportMode.UDS
PyAEDT INFO: Electronics Desktop started on gRPC port 42571 after 12.9 seconds.
PyAEDT INFO: AEDT installation Path /home/harry/.local/share/ansysedt-podman/v252/AnsysEM
PyAEDT INFO: Connec